In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root (the directory holding research_notebooks/ and the
# Makefile) so repo-root-relative CONFIG_PATH values resolve regardless of how
# the notebook was launched (jupyter CWD = notebook dir, scheduler = repo root).
_repo_root = _lab_root
for _candidate in [_lab_root, *_lab_root.parents]:
    if (_candidate / "research_notebooks").is_dir() and (_candidate / "Makefile").is_file():
        _repo_root = _candidate
        break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 04 — Intraday Event Replay

In [ ]:
import datetime as _dt
import pandas as pd
from pathlib import Path
from bowaka_v2_lab.config import load_config
from bowaka_v2_lab.scanner.replay import replay_scanner
from bowaka_v2_lab.sim.replay_fixtures import synthetic_universe, synthetic_daily_cache
cfg = load_config(CONFIG_PATH)
syms = cfg.get('universe', {}).get('symbols') or ['AAA','BBB','CCC']
universe = synthetic_universe(syms)
daily_cache = synthetic_daily_cache(syms)
scan_ts = [pd.Timestamp('2024-09-04 14:00:00', tz='UTC')]
_md = cfg.get('market_data', {})
if _md.get('minute_bar_source', 'fixture') in ('alpaca', 'shared'):
    # Real data: bars come from the shared market-data lake.
    from bowaka_v2_lab.data.suppliers import make_lake_suppliers
    supplier, _ = make_lake_suppliers(_md.get('shared_root'), feed=_md.get('feed', 'iex'))
else:
    def supplier(sym, ts):
        rows = []
        for i in range(30):
            rows.append({'timestamp': pd.Timestamp('2024-09-04 13:30:00', tz='UTC') + pd.Timedelta(minutes=i),
                          'open': 100+i*0.1, 'high': 100+i*0.2, 'low': 99.5, 'close': 100+i*0.15, 'volume': 1000.0})
        return pd.DataFrame(rows)
run_dir = Path('research_notebooks/bowaka_v2_lab/artifacts/runs/replay_nb_smoke')
summary = replay_scanner(cfg=cfg, universe_snapshot=universe, daily_cache=daily_cache,
  volume_curve=None, scan_timestamps=scan_ts, bars_supplier=supplier, run_dir=run_dir)
print(summary)
